# Step 4 : Expansion de la KB via SPARQL

On interroge Wikidata pour passer de ~1700 à 50 000+ triplets.

Strategie en 5 niveaux :
1. Proprietes directes de nos 45 artistes
2. Albums et singles de nos 45 artistes
3. Artistes lies par genre
4. Proprietes completes des artistes lies
5. Albums des artistes lies

In [1]:
import subprocess
subprocess.run(["pip", "install", "urllib3==1.26.18", "requests==2.31.0", "--quiet", "--force-reinstall"])
print(" OK")

 OK


In [2]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD
import requests, time, json, re
from urllib.parse import quote

EX  = Namespace("http://musickg.example.org/resource/")
EXO = Namespace("http://musickg.example.org/ontology/")
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

CHEMIN   = "C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/"
ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS  = {"User-Agent": "MusicKG-Student/1.0", "Accept": "application/json"}

# Mapping predicats Wikidata -> predicats de notre KB
PROP_MAP = {
    "P136":  EXO.hasGenre,        "P264":  EXO.signedTo,
    "P495":  EXO.originCountry,   "P737":  EXO.influencedBy,
    "P166":  EXO.hasAward,        "P463":  EXO.memberOf,
    "P19":   EXO.birthPlace,      "P27":   EXO.citizenship,
    "P1303": EXO.playsInstrument, "P412":  EXO.voiceType,
    "P21":   EXO.gender,          "P1412": EXO.languageUsed,
    "P175":  EXO.performedBy,     "P577":  EXO.releaseYear,
    "P571":  EXO.activeFrom,      "P576":  EXO.activeTo,
    "P569":  EXO.birthDate,       "P570":  EXO.deathDate,
    "P740":  EXO.formationPlace,  "P527":  EXO.hasMember,
    "P162":  EXO.producedBy,      "P358":  EXO.hasDiscography,
}

def query_wikidata(q, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(ENDPOINT, params={"query": q, "format": "json"}, headers=HEADERS, timeout=30)
            if r.status_code == 200:
                return r.json().get("results", {}).get("bindings", [])
            elif r.status_code == 429:
                time.sleep(10 * (attempt + 1))
        except:
            time.sleep(5)
    return []

def safe_uri(name):
    clean = re.sub(r"[^\w\s\-]", "", str(name)).strip()
    clean = re.sub(r"\s+", "_", clean)
    return EX[quote(clean, safe="_-")]

def expand_entity(uri, qid, limit=300):
    """Recupere toutes les proprietes d'une entite Wikidata."""
    query = f"""
SELECT ?prop ?val ?valLabel WHERE {{
  wd:{qid} ?prop ?val .
  FILTER(STRSTARTS(STR(?prop), "http://www.wikidata.org/prop/direct/"))
  OPTIONAL {{
    ?val rdfs:label ?valLabel .
    FILTER(LANG(?valLabel) IN ("fr", "en"))
  }}
}} LIMIT {limit}
"""
    for row in query_wikidata(query):
        prop_id   = row["prop"]["value"].split("/")[-1]
        val       = row["val"]["value"]
        val_label = row.get("valLabel", {}).get("value", "")
        predicat  = PROP_MAP.get(prop_id, WDT[prop_id])
        if row["val"]["type"] == "uri":
            objet = URIRef(val)
            if val_label:
                g.add((objet, RDFS.label, Literal(val_label[:200], lang="fr")))
        else:
            objet = Literal(val[:500] if len(val) > 500 else val)
        g.add((uri, predicat, objet))

print("Tout est pret.")

Tout est pret.


### Chargement de la KB alignee

In [3]:
g = Graph()
g.bind("ex", EX); g.bind("exo", EXO); g.bind("wd", WD)
g.bind("wdt", WDT); g.bind("owl", OWL); g.bind("rdfs", RDFS)

g.parse(CHEMIN + "aligned_kb.ttl", format="turtle")

artistes_qids = {}
for s, _, o in g.triples((None, OWL.sameAs, None)):
    o_str = str(o)
    if "wikidata.org/entity/Q" in o_str:
        qid = o_str.split("/")[-1]
        nom = str(s).split("/")[-1].replace("_", " ")
        artistes_qids[nom] = qid

print(f"KB chargee        : {len(g)} triplets")
print(f"Artistes a traiter : {len(artistes_qids)}")

KB chargee        : 1918 triplets
Artistes a traiter : 48


### Niveau 1 : toutes les proprietes de nos 45 artistes

Pour chaque artiste aligne, on recupere TOUTES ses proprietes Wikidata
en une seule requete. C'est plus efficace que de demander propriete par propriete.

In [4]:
t0 = len(g)
for i, (nom, qid) in enumerate(artistes_qids.items()):
    expand_entity(safe_uri(nom), qid, limit=300)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(artistes_qids)} artistes | triplets : {len(g)}")
    time.sleep(1.5)

print(f"\nTriplets ajoutes niveau 1 : {len(g) - t0}")
print(f"Total : {len(g)}")

  10/48 artistes | triplets : 3700
  20/48 artistes | triplets : 5517
  30/48 artistes | triplets : 6981
  40/48 artistes | triplets : 7941

Triplets ajoutes niveau 1 : 7691
Total : 9609


### Niveau 2 : albums et singles de nos artistes

In [5]:
t0 = len(g)

for nom, qid in artistes_qids.items():
    uri_artiste = safe_uri(nom)

    query = f"""
SELECT DISTINCT ?item ?prop ?val ?valLabel WHERE {{
  ?item wdt:P175 wd:{qid} .
  ?item wdt:P31 ?type .
  FILTER(?type IN (wd:Q482994, wd:Q169930, wd:Q7366, wd:Q134556, wd:Q208569))
  ?item ?prop ?val .
  FILTER(STRSTARTS(STR(?prop), "http://www.wikidata.org/prop/direct/"))
  OPTIONAL {{
    ?val rdfs:label ?valLabel .
    FILTER(LANG(?valLabel) IN ("fr", "en"))
  }}
}} LIMIT 2000
"""
    for row in query_wikidata(query):
        item_uri  = URIRef(row["item"]["value"])
        prop_id   = row["prop"]["value"].split("/")[-1]
        val       = row["val"]["value"]
        val_label = row.get("valLabel", {}).get("value", "")

        g.add((uri_artiste, EXO.hasWork,  item_uri))
        g.add((item_uri,    RDF.type,     EXO.MusicalWork))

        predicat = PROP_MAP.get(prop_id, WDT[prop_id])
        if row["val"]["type"] == "uri":
            objet = URIRef(val)
            if val_label:
                g.add((objet, RDFS.label, Literal(val_label[:200], lang="fr")))
        else:
            objet = Literal(val[:500] if len(val) > 500 else val)
        g.add((item_uri, predicat, objet))

    time.sleep(1.5)

print(f"Triplets ajoutes niveau 2 : {len(g) - t0}")
print(f"Total : {len(g)}")

Triplets ajoutes niveau 2 : 25681
Total : 35290


### Niveau 3 : artistes lies par genre

In [6]:
t0 = len(g)
artistes_lies_qids = {}

for nom, qid in artistes_qids.items():
    uri_artiste = safe_uri(nom)

    query = f"""
SELECT DISTINCT ?artiste ?artisteLabel WHERE {{
  wd:{qid} wdt:P136 ?genre .
  ?artiste wdt:P136 ?genre .
  ?artiste wdt:P31 wd:Q5 .
  FILTER(?artiste != wd:{qid})
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "fr,en" . }}
}} LIMIT 30
"""
    for row in query_wikidata(query):
        a_uri   = URIRef(row["artiste"]["value"])
        a_qid   = row["artiste"]["value"].split("/")[-1]
        a_label = row.get("artisteLabel", {}).get("value", "")

        g.add((uri_artiste, EXO.sameGenreAs, a_uri))
        g.add((a_uri, RDF.type, EXO.Artist))
        if a_label:
            g.add((a_uri, RDFS.label, Literal(a_label[:200], lang="fr")))

        if a_qid not in artistes_qids.values():
            artistes_lies_qids[a_qid] = a_uri

    time.sleep(1.5)

print(f"Triplets ajoutes niveau 3 : {len(g) - t0}")
print(f"Total : {len(g)}")
print(f"Nouveaux artistes a expander : {len(artistes_lies_qids)}")

Triplets ajoutes niveau 3 : 1475
Total : 36765
Nouveaux artistes a expander : 468


### Niveau 4 : proprietes completes des artistes lies

In [7]:
t0 = len(g)
total_lies = len(artistes_lies_qids)

for i, (qid, uri) in enumerate(artistes_lies_qids.items()):
    expand_entity(uri, qid, limit=200)
    if (i+1) % 20 == 0:
        print(f"  {i+1}/{total_lies} | triplets : {len(g)}")
    time.sleep(1)

print(f"\nTriplets ajoutes niveau 4 : {len(g) - t0}")
print(f"Total : {len(g)}")

  20/468 | triplets : 38578
  40/468 | triplets : 40403
  60/468 | triplets : 43136
  80/468 | triplets : 45146
  100/468 | triplets : 46657
  120/468 | triplets : 48682
  140/468 | triplets : 50544
  160/468 | triplets : 52475
  180/468 | triplets : 53981
  200/468 | triplets : 55729
  220/468 | triplets : 57122
  240/468 | triplets : 58887
  260/468 | triplets : 60445
  280/468 | triplets : 62574
  300/468 | triplets : 65211
  320/468 | triplets : 66925
  340/468 | triplets : 69349
  360/468 | triplets : 71349
  380/468 | triplets : 72690
  400/468 | triplets : 75045
  420/468 | triplets : 77995
  440/468 | triplets : 80524
  460/468 | triplets : 82353

Triplets ajoutes niveau 4 : 46644
Total : 83409


### Niveau 5 : albums des artistes lies

In [8]:
t0 = len(g)

for i, (qid, uri) in enumerate(artistes_lies_qids.items()):
    query = f"""
SELECT DISTINCT ?item ?prop ?val ?valLabel WHERE {{
  ?item wdt:P175 wd:{qid} .
  ?item wdt:P31 ?type .
  FILTER(?type IN (wd:Q482994, wd:Q169930, wd:Q7366, wd:Q134556))
  ?item ?prop ?val .
  FILTER(STRSTARTS(STR(?prop), "http://www.wikidata.org/prop/direct/"))
  OPTIONAL {{
    ?val rdfs:label ?valLabel .
    FILTER(LANG(?valLabel) IN ("fr", "en"))
  }}
}} LIMIT 500
"""
    for row in query_wikidata(query):
        item_uri  = URIRef(row["item"]["value"])
        prop_id   = row["prop"]["value"].split("/")[-1]
        val       = row["val"]["value"]
        val_label = row.get("valLabel", {}).get("value", "")

        g.add((uri, EXO.hasWork, item_uri))
        g.add((item_uri, RDF.type, EXO.MusicalWork))

        predicat = PROP_MAP.get(prop_id, WDT[prop_id])
        if row["val"]["type"] == "uri":
            objet = URIRef(val)
            if val_label:
                g.add((objet, RDFS.label, Literal(val_label[:200], lang="fr")))
        else:
            objet = Literal(val[:500] if len(val) > 500 else val)
        g.add((item_uri, predicat, objet))

    if (i+1) % 20 == 0:
        print(f"  {i+1}/{len(artistes_lies_qids)} | triplets : {len(g)}")
    time.sleep(1)

print(f"\nTriplets ajoutes niveau 5 : {len(g) - t0}")
print(f"Total : {len(g)}")

  20/468 | triplets : 86450
  40/468 | triplets : 89285
  60/468 | triplets : 93196
  80/468 | triplets : 95481
  100/468 | triplets : 96942
  120/468 | triplets : 99684
  140/468 | triplets : 101828
  160/468 | triplets : 104899
  180/468 | triplets : 105998
  200/468 | triplets : 109169
  220/468 | triplets : 111259
  240/468 | triplets : 112156
  260/468 | triplets : 113735
  280/468 | triplets : 114558
  300/468 | triplets : 114688
  320/468 | triplets : 117765
  340/468 | triplets : 121033
  360/468 | triplets : 124327
  380/468 | triplets : 125159
  400/468 | triplets : 128946
  420/468 | triplets : 133156
  440/468 | triplets : 138045
  460/468 | triplets : 140745

Triplets ajoutes niveau 5 : 58840
Total : 142249


In [12]:
from collections import Counter

print(f"Triplets avant nettoyage strict : {len(g)}")

# 1. Compter la fréquence de tous les prédicats dans le graphe
compteur_predicats = Counter(g.predicates())

# 2. Conserver uniquement les 150 prédicats les plus utilisés
# (Cela garantit de respecter la consigne de 50 à 200 relations)
top_predicats = set(p for p, count in compteur_predicats.most_common(150))

# 3. Trouver tous les triplets qui utilisent un prédicat hors de ce top 150
triplets_a_supprimer = []
for s, p, o in g:
    if p not in top_predicats:
        triplets_a_supprimer.append((s, p, o))

# 4. Supprimer ces triplets du graphe
for t in triplets_a_supprimer:
    g.remove(t)

print(f"Triplets supprimés : {len(triplets_a_supprimer)}")
print(f"Total de triplets après nettoyage : {len(g)}")
print(f"Nouveau nombre de prédicats uniques : {len(set(g.predicates()))}")

Triplets avant nettoyage strict : 140336
Triplets supprimés : 15308
Total de triplets après nettoyage : 125028
Nouveau nombre de prédicats uniques : 150


### Nettoyage avant export KGE

On supprime les predicats inutiles pour l'entrainement KGE
(URLs d'images, IDs externes, coordonnees GPS...).

In [13]:
PREDICATS_A_SUPPRIMER = [
    WDT["P18"],   WDT["P856"],  WDT["P2397"],
    WDT["P345"],  WDT["P434"],  WDT["P625"],
    WDT["P973"],  WDT["P854"],  WDT["P813"],
    WDT["P143"],  WDT["P4656"],
]

t0 = len(g)
for pred in PREDICATS_A_SUPPRIMER:
    for t in list(g.triples((None, pred, None))):
        g.remove(t)

print(f"Triplets supprimes (predicats inutiles) : {t0 - len(g)}")
print(f"Total apres nettoyage : {len(g)}")

Triplets supprimes (predicats inutiles) : 0
Total apres nettoyage : 125028


### Statistiques finales

In [14]:
total     = len(g)
entites   = len(set(g.subjects()) | set(o for _, _, o in g if isinstance(o, URIRef)))
predicats = len(set(g.predicates()))
oeuvres   = len(list(g.subjects(RDF.type, EXO.MusicalWork)))
artistes  = len(list(g.subjects(RDF.type, EXO.Artist)))

print("STATISTIQUES FINALES")
print(f"Triplets totaux    : {total}")
print(f"Entites uniques    : {entites}")
print(f"Predicats uniques  : {predicats}")
print(f"Oeuvres musicales  : {oeuvres}")
print(f"Artistes lies      : {artistes}")
print()
print("Verification des objectifs :")
print(f"  Triplets  50k-200k  : {50000 <= total <= 200000}   ({total})")
print(f"  Entites   5k-30k    : {5000 <= entites <= 30000}   ({entites})")
print(f"  Predicats 50-200    : {50 <= predicats <= 200}   ({predicats})")

STATISTIQUES FINALES
Triplets totaux    : 125028
Entites uniques    : 19768
Predicats uniques  : 150
Oeuvres musicales  : 5713
Artistes lies      : 471

Verification des objectifs :
  Triplets  50k-200k  : True   (125028)
  Entites   5k-30k    : True   (19768)
  Predicats 50-200    : True   (150)


### Sauvegarde

In [15]:
g.serialize(CHEMIN + "expanded_kb.ttl", format="turtle")
print("Fichier sauvegarde : expanded_kb.ttl")

g.serialize(CHEMIN + "expanded_kb.nt", format="ntriples")
print("Fichier sauvegarde : expanded_kb.nt")

stats = {
    "triplets_totaux":   len(g),
    "entites_uniques":   len(set(g.subjects()) | set(o for _, _, o in g if isinstance(o, URIRef))),
    "predicats_uniques": len(set(g.predicates())),
    "oeuvres_musicales": len(list(g.subjects(RDF.type, EXO.MusicalWork))),
    "artistes_lies":     len(list(g.subjects(RDF.type, EXO.Artist))),
}
with open(CHEMIN + "kb_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)
print("Fichier sauvegarde : kb_stats.json")

Fichier sauvegarde : expanded_kb.ttl
Fichier sauvegarde : expanded_kb.nt
Fichier sauvegarde : kb_stats.json
